In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import make_scorer, accuracy_score, f1_score
from time import time

DATASET_LOCATION = "C:/Users/vasilis/Desktop/machineLearning/Twitter_US_Airline_Sentiment.csv"

# Loading the dataset
data = pd.read_csv(DATASET_LOCATION, sep=',')  

# Examine the dataset
print(data.head())

# Extract features and labels
X = data['text']  # The column containing text data
y = data['airline_sentiment']  # The target labels (positive, neutral, negative)

# Function that preprocessing the data , using TfidfVectorizer
def preprocess_tfidf(X, min_df=None, max_features=None):
    vectorizer = TfidfVectorizer(min_df=min_df, max_features=max_features)
    return vectorizer.fit_transform(X)


# Classifiers
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Support Vector Machines": LinearSVC(),
    "Random Forests": RandomForestClassifier(),
    "Neural Network": MLPClassifier(max_iter=1000)
}

# Function that evaluates classifiers using cross-validation
def evaluate_classifiers(X, y, classifiers):
    results = {}
    metrics = {
        'accuracy': make_scorer(accuracy_score),
        'f1_score': make_scorer(f1_score, average='weighted'),
    }

    for name, clf in classifiers.items():
        scores = cross_validate(clf, X, y, cv=5, scoring=metrics, return_train_score=False)
        results[name] = {
            'Accuracy': np.mean(scores['test_accuracy']),
            'F1-score': np.mean(scores['test_f1_score']),
            'Fit time': np.mean(scores['fit_time'])  # Automatically included in cross_validate
        }
    return results

# Case 1: Words appearing in at least 5 documents
print("Running Case 1: Words appearing in at least 5 documents")
X_tfidf_case1 = preprocess_tfidf(X, min_df=5)
results_case1 = evaluate_classifiers(X_tfidf_case1, y, classifiers)

# Case 2: Top 2500 words
print("Running Case 2: Top 2500 words")
X_tfidf_case2 = preprocess_tfidf(X, max_features=2500)
results_case2 = evaluate_classifiers(X_tfidf_case2, y, classifiers)

# Case 3: Top 500 words
print("Running Case 3: Top 500 words")
X_tfidf_case3 = preprocess_tfidf(X, max_features=500)
results_case3 = evaluate_classifiers(X_tfidf_case3, y, classifiers)

# Consolidate and display results
results = {
    "Case 1 (min_df=5)": results_case1,
    "Case 2 (max_features=2500)": results_case2,
    "Case 3 (max_features=500)": results_case3
}

for case, res in results.items():
    print(f"\n{case}")
    for clf_name, metrics in res.items():
        print(f"{clf_name}: {metrics}")

             tweet_id airline_sentiment  airline_sentiment_confidence  \
0  570306133677760513           neutral                        1.0000   
1  570301130888122368          positive                        0.3486   
2  570301083672813571           neutral                        0.6837   
3  570301031407624196          negative                        1.0000   
4  570300817074462722          negative                        1.0000   

  negativereason  negativereason_confidence         airline  \
0            NaN                        NaN  Virgin America   
1            NaN                     0.0000  Virgin America   
2            NaN                        NaN  Virgin America   
3     Bad Flight                     0.7033  Virgin America   
4     Can't Tell                     1.0000  Virgin America   

  airline_sentiment_gold        name negativereason_gold  retweet_count  \
0                    NaN     cairdin                 NaN              0   
1                    NaN    jnar